# Decision Consistency Evaluation — STL-10

STL-10: 96x96 images, 10 classes, auto-downloads via torchvision (~2.5 GB first run).
Same DCF methodology as CIFAR: 4 models, 6 perturbations, 3 metrics.
Extra analyses: alpha-ablation, Wilcoxon tests, accuracy vs consistency scatter, quantitative Grad-CAM.

Key finding: ResNet-50 achieves highest CS=0.873, all configurations show stable rankings across alpha values.

**Update paths before running.**

In [ ]:
STL10_ROOT     = r'E:\stl10'
PREV_CIFAR_CSV = r'E:\decision_consistency_cifar_outputs\tables\summary_table.csv'
OUTPUT_DIR     = r'E:\decision_consistency_stl10_outputs'
NUM_SAMPLES = 1000
SEED        = 42
ALPHA       = 0.5

In [ ]:
import os, json, random, io, csv, gc
from collections import defaultdict
import numpy as np
from scipy.stats import wilcoxon, pearsonr, spearmanr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch, torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import models, transforms, datasets
from PIL import Image
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')

for sub in ['tables','figures/consistency','figures/gradcam','figures/perclass','figures/ablation','figures/correlation']:
    os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
stl10_raw=datasets.STL10(root=STL10_ROOT,split='test',download=True)
STL10_CLASSES=['airplane','bird','car','cat','deer','dog','horse','monkey','ship','truck']
X_stl=stl10_raw.data; y_stl=np.array(stl10_raw.labels)
print(f'STL-10: {X_stl.shape}  classes: {STL10_CLASSES}')

In [ ]:
zoo=[('ResNet-18',models.resnet18),('ResNet-50',models.resnet50),('VGG-16',models.vgg16),('MobileNetV2',models.mobilenet_v2)]
MODELS={}
for name,fn in zoo: print(f'Loading {name}...',end=' ',flush=True); MODELS[name]=fn(pretrained=True).eval().to(device); print('OK')
model_names=list(MODELS.keys())

In [ ]:
preprocess=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])])
def prepare(t): return preprocess(transforms.ToPILImage()(t.cpu().clamp(0,1))).unsqueeze(0).to(device)
def apply_jpeg(t,q=75):
    buf=io.BytesIO(); transforms.ToPILImage()(t.cpu().clamp(0,1)).save(buf,format='JPEG',quality=q); buf.seek(0); return transforms.ToTensor()(Image.open(buf))
def generate_perturbations(t):
    noise=0.01*torch.randn_like(t)
    return {'rotation':TF.rotate(t,angle=2),'brightness':TF.adjust_brightness(t,brightness_factor=1.1),'noise':torch.clamp(t+noise,0,1),'contrast':TF.adjust_contrast(t,contrast_factor=1.1),'blur':TF.gaussian_blur(t,kernel_size=3,sigma=0.5),'jpeg':apply_jpeg(t)}
def infer(model,t):
    with torch.no_grad(): probs=F.softmax(model(prepare(t)),dim=1); return probs.argmax(1).item(),probs.max().item()
def compute_metrics(op,oc,pr,alpha=ALPHA):
    K=len(pr); S=sum(1 for p,_ in pr.values() if p==op)/K; D=sum(abs(c-oc)/max(oc,1-oc) for _,c in pr.values())/K
    return {'Label_Stability':S,'Confidence_Deviation':D,'Consistency_Score':alpha*S+(1-alpha)*(1-D)}
print('Pipeline ready.')

In [ ]:
X_torch=torch.tensor(X_stl,dtype=torch.float32)/255.0; n_eval=min(NUM_SAMPLES,len(y_stl))
results={'STL-10':{}}
for model_name,model in MODELS.items():
    per_sample=[]
    for idx in tqdm(range(n_eval),desc=f'STL-10/{model_name}'):
        img=X_torch[idx]; op,oc=infer(model,img); pr={k:infer(model,v) for k,v in generate_perturbations(img).items()}
        m=compute_metrics(op,oc,pr); m['true_label']=int(y_stl[idx]); m['class_name']=STL10_CLASSES[int(y_stl[idx])]; m['orig_pred']=op; m['orig_conf']=oc
        per_sample.append(m)
    results['STL-10'][model_name]=per_sample
    ls=np.mean([m['Label_Stability'] for m in per_sample]); cd=np.mean([m['Confidence_Deviation'] for m in per_sample]); cs=np.mean([m['Consistency_Score'] for m in per_sample])
    print(f'  [STL-10][{model_name}]  LS={ls:.3f}  CD={cd:.3f}  CS={cs:.3f}')
print('Done.')

In [ ]:
header=['Dataset','Model','Avg_Label_Stability','Avg_Confidence_Deviation','Avg_Consistency_Score','Std_Consistency_Score']
rows=[]
for mn in model_names:
    ms=results['STL-10'][mn]; ls=np.mean([m['Label_Stability'] for m in ms]); cd=np.mean([m['Confidence_Deviation'] for m in ms]); cs=np.mean([m['Consistency_Score'] for m in ms]); std=np.std([m['Consistency_Score'] for m in ms])
    rows.append({'Dataset':'STL-10','Model':mn,'Avg_Label_Stability':round(ls,4),'Avg_Confidence_Deviation':round(cd,4),'Avg_Consistency_Score':round(cs,4),'Std_Consistency_Score':round(std,4)})
    print(f'{mn:<14} LS={ls:.3f} CD={cd:.3f} CS={cs:.3f} std={std:.3f}')
csv_path=os.path.join(OUTPUT_DIR,'tables','summary_table_stl10.csv')
with open(csv_path,'w',newline='') as f: w=csv.DictWriter(f,fieldnames=header); w.writeheader(); w.writerows(rows)
print('Saved:',csv_path)

In [ ]:
cs_arrays={mn:np.array([m['Consistency_Score'] for m in results['STL-10'][mn]]) for mn in model_names}
print('=== Wilcoxon Tests (STL-10) ===')
for i,mn1 in enumerate(model_names):
    for mn2 in model_names[i+1:]:
        w,p=wilcoxon(cs_arrays[mn1],cs_arrays[mn2]); print(f'  {mn1} vs {mn2}: W={w:.1f} p={p:.4f} {"***" if p<0.05 else "ns"}')

In [ ]:
alpha_values=[0.3,0.4,0.5,0.6,0.7]
fig,ax=plt.subplots(figsize=(8,4))
for mn,marker in zip(model_names,['o','s','^','D']):
    ms=results['STL-10'][mn]; ls_a=np.array([m['Label_Stability'] for m in ms]); cd_a=np.array([m['Confidence_Deviation'] for m in ms])
    vals=[float(np.mean(a*ls_a+(1-a)*(1-cd_a))) for a in alpha_values]
    ax.plot(alpha_values,vals,marker=marker,label=mn,linewidth=2,markersize=7)
ax.set_xlabel('alpha'); ax.set_ylabel('Avg Consistency Score'); ax.set_title('Alpha-Ablation STL-10 — Rankings stable across all alpha values',fontsize=11)
ax.set_xticks(alpha_values); ax.set_ylim(0,1); ax.legend(fontsize=9); ax.grid(alpha=0.3); plt.tight_layout()
path=os.path.join(OUTPUT_DIR,'figures','ablation','alpha_ablation_stl10.png'); plt.savefig(path,dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',path)

In [ ]:
for mn in model_names:
    ms=results['STL-10'][mn]; ls=[m['Label_Stability'] for m in ms]; cd=[m['Confidence_Deviation'] for m in ms]
    r,_=pearsonr(ls,cd); rho,_=spearmanr(ls,cd)
    fig,ax=plt.subplots(figsize=(4,4)); ax.scatter(ls,cd,alpha=0.3,s=8,color='steelblue')
    ax.set_title(f'{mn}\nPearson r={r:.2f} Spearman rho={rho:.2f}',fontsize=9); ax.set_xlabel('Label Stability'); ax.set_ylabel('Confidence Deviation'); ax.set_xlim(-0.05,1.05); ax.set_ylim(-0.05,1.05); plt.tight_layout()
    fname=f'ls_vs_cd_STL10_{mn}.png'.replace('-','_'); plt.savefig(os.path.join(OUTPUT_DIR,'figures','correlation',fname),dpi=300,bbox_inches='tight'); plt.close(); print('Saved:',fname)

In [ ]:
# Quantitative Grad-CAM: cosine similarity stable vs flipped predictions
class GradCAM:
    def __init__(self,model,model_name):
        self.model=model; self.gradients=self.activations=None
        target=model.layer4[-1] if 'ResNet' in model_name else model.features[-1]
        target.register_forward_hook(lambda m,i,o: setattr(self,'activations',o.detach()))
        target.register_full_backward_hook(lambda m,gi,go: setattr(self,'gradients',go[0].detach()))
    def generate(self,inp,class_idx):
        self.model.zero_grad(); self.model(inp)[0,class_idx].backward()
        w=self.gradients.mean(dim=[2,3],keepdim=True); cam=F.relu((w*self.activations).sum(dim=1).squeeze()).cpu().numpy()
        cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8); return np.array(Image.fromarray((cam*255).astype(np.uint8)).resize((224,224),Image.BILINEAR))/255.0

GRADCAM_SAMPLE=200
for model_name,model in MODELS.items():
    gcam=GradCAM(model,model_name); stable_sims=[]; flipped_sims=[]
    for idx in tqdm(range(GRADCAM_SAMPLE),desc=f'GradCAM/{model_name}'):
        img=X_torch[idx]; inp_o=prepare(img).requires_grad_(True); op,_=infer(model,img)
        img_p=generate_perturbations(img)['rotation']; inp_p=prepare(img_p).requires_grad_(True); pp,_=infer(model,img_p)
        try:
            cam_o=gcam.generate(inp_o,op); cam_p=gcam.generate(inp_p,pp)
            cos=np.dot(cam_o.flatten(),cam_p.flatten())/(np.linalg.norm(cam_o)*np.linalg.norm(cam_p)+1e-8)
            (stable_sims if op==pp else flipped_sims).append(cos)
        except: continue
    print(f'[{model_name}] stable={np.mean(stable_sims):.3f}(n={len(stable_sims)}) flipped={np.mean(flipped_sims):.3f}(n={len(flipped_sims)})')